# University Intent Classification
Fine-tuning `bert-base-uncased` for student query routing.

**Run cells top to bottom. After training, the model is saved to `../model/`.**

## 1. Install Dependencies

In [4]:
# Run this once, then restart kernel if needed
# !pip install transformers torch datasets scikit-learn pandas matplotlib seaborn

In [5]:
# import subprocess, sys

# packages = [
#     "transformers",
#     "torch",
#     "datasets",
#     "scikit-learn",
#     "pandas",
#     "matplotlib",
#     "seaborn",
#     "streamlit",
# ]

# subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

In [ ]:
import os
print(os.getcwd())

## 2. Imports

In [6]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Matplotlib is building the font cache; this may take a moment.
/Users/kevinartan/Documents/GitHub/school_dashboard_administration/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## 3. Load & Explore Data

In [7]:
TRAIN_PATH = Path.cwd().parent / "dataset" / "university_query_train.csv"
TEST_PATH = Path.cwd().parent / "dataset" / "university_query_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# TRAIN_PATH = '../dataset/university_query_train.csv'
# TEST_PATH  = '../dataset/university_query_test.csv'

# train_df = pd.read_csv(TRAIN_PATH)
# test_df  = pd.read_csv(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test shape: ', test_df.shape)
print()
print('Train columns:', train_df.columns.tolist())
train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/kevinartan/Documents/GitHub/dataset/university_query_train.csv'

In [ ]:
# ── Adjust these two lines to match your actual column names ──────────────
TEXT_COL  = 'query'     # column containing the student question
LABEL_COL = 'category'  # column containing the office/intent label
# ─────────────────────────────────────────────────────────────────────────

# Auto-detect if the defaults don't exist
if TEXT_COL not in train_df.columns or LABEL_COL not in train_df.columns:
    print('Column names found:', train_df.columns.tolist())
    print('Update TEXT_COL and LABEL_COL above to match!')
else:
    print('Labels found in dataset:')
    print(train_df[LABEL_COL].value_counts())

In [ ]:
# Label distribution plot
plt.figure(figsize=(10, 4))
train_df[LABEL_COL].value_counts().plot(kind='bar', color='steelblue')
plt.title('Label Distribution (Train)')
plt.xlabel('Office / Intent')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# Encode string labels → integers
le = LabelEncoder()
train_df['label_id'] = le.fit_transform(train_df[LABEL_COL])
test_df['label_id']  = le.transform(test_df[LABEL_COL])

NUM_LABELS = len(le.classes_)
print(f'Number of classes: {NUM_LABELS}')
print('Class mapping:')
for i, c in enumerate(le.classes_):
    print(f'  {i} → {c}')

In [ ]:
# Drop rows with missing text
train_df = train_df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
test_df  = test_df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
print('After dropping NaN — Train:', len(train_df), '| Test:', len(test_df))

## 5. Dataset & DataLoader

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 128
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.tolist()
        self.labels    = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IntentDataset(train_df[TEXT_COL], train_df['label_id'], tokenizer, MAX_LEN)
test_dataset  = IntentDataset(test_df[TEXT_COL],  test_df['label_id'],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 6. Model Setup

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)
model = model.to(DEVICE)

EPOCHS    = 4
LR        = 2e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f'Total training steps: {total_steps}')

## 7. Training

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds  = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds  = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    return total_loss / len(loader), correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
    vl_loss, vl_acc = eval_epoch(model, test_loader, DEVICE)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    print(f'Epoch {epoch}/{EPOCHS} '
          f'| train loss {tr_loss:.4f} acc {tr_acc:.4f} '
          f'| val loss {vl_loss:.4f} acc {vl_acc:.4f}')

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_
))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 10. Save Model + Label Mapping

In [ ]:
SAVE_DIR = '../model'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save BERT weights + tokenizer
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save label mapping so app.py can decode predictions
label_map = {str(i): label for i, label in enumerate(le.classes_)}
with open(os.path.join(SAVE_DIR, 'label_map.json'), 'w') as f:
    json.dump(label_map, f, indent=2)

print('Saved to', SAVE_DIR)
print('Label map:', label_map)

## 11. Quick Sanity Check
Test a few example queries before launching the app.

In [ ]:
def predict(text, model, tokenizer, label_map, device, max_len=128):
    model.eval()
    enc = tokenizer(
        text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    with torch.no_grad():
        outputs = model(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device)
        )
    pred_id = outputs.logits.argmax(dim=1).item()
    probs   = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()
    return label_map[str(pred_id)], float(probs[pred_id])


sample_queries = [
    'I have not received my scholarship payment yet',
    'My student email account is not working',
    'I want to request a transcript',
    'When is the deadline to add or drop a course?',
    'The WiFi in the dorm keeps disconnecting',
]

for q in sample_queries:
    intent, conf = predict(q, model, tokenizer, label_map, DEVICE)
    print(f'  [{conf:.0%}] "{q}"')
    print(f'        → {intent}\n')